# arXiv Harvester trên Kaggle

Notebook thu thập dữ liệu M1, hỗ trợ checkpoint và đồng bộ Hugging Face.

Cấu hình trước khi chạy:

- Accelerator: `None`; Internet: bật.
- Persistence: `Files only` hoặc `Variables and Files`.
- Secrets: `HF_KEY` cho upload; `S2_API_KEY` nếu có.
- Chọn `CURRENT_MEMBER`: `DAT`, `TUAN`, `TAN`, `TRI` hoặc `TRIET`.

Crawler tải từng version, chỉ giữ `.tex/.bib`, giãn request arXiv tối thiểu 3 giây và xử lý `Retry-After`. References lấy từ S2; BibTeX local được dùng khi S2 trả 404. PDF-only vẫn giữ metadata.

[Yêu cầu đồ án](https://hcmus-cs-ta.github.io/lab-book/intro-ds/statement/crawling/).


In [ ]:
# Cấu hình thành viên và crawler.
import os
import sys

# Chọn DAT, TUAN, TAN, TRI hoặc TRIET.
CURRENT_MEMBER: str = "DAT"

# Hugging Face và S2 credentials.
MANUAL_HF_KEY: str = ""      # Dán token hf_... vào đây nếu không dùng Kaggle Secrets
MANUAL_S2_API_KEY: str = ""  # Dán Semantic Scholar API Key vào đây (nếu có)

try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    HF_KEY: str = user_secrets.get_secret("HF_KEY") or MANUAL_HF_KEY
    print("Đã tải HF_KEY thành công từ Kaggle Secrets.")
except Exception:
    HF_KEY: str = MANUAL_HF_KEY or os.environ.get("HF_KEY", "")
    if HF_KEY:
        print("Đã tải HF_KEY từ cấu hình thủ công.")
    else:
        print("Thiếu HF_KEY; chưa thể đồng bộ Hugging Face.")

try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    S2_API_KEY: str = user_secrets.get_secret("S2_API_KEY") or MANUAL_S2_API_KEY
    if S2_API_KEY:
        os.environ["S2_API_KEY"] = S2_API_KEY
        print("Đã tải S2_API_KEY từ Kaggle Secrets.")
except Exception:
    S2_API_KEY = MANUAL_S2_API_KEY or os.environ.get("S2_API_KEY", "")
    if S2_API_KEY:
        os.environ["S2_API_KEY"] = S2_API_KEY
        print("Đã tải S2_API_KEY từ cấu hình thủ công.")
    else:
        print("Semantic Scholar: public API.")

# Dải ID của từng thành viên.
MEMBERS_CONFIG = {
    "DAT": {
        "name": "Nguyễn Quốc Đạt",
        "member_id": "24120282",
        "start_month": "2024-12",
        "start_id": 18848,
        "end_month": "2025-01",
        "end_id": 2641,
    },
    "TUAN": {
        "name": "Nguyễn Thành Tuấn",
        "member_id": "24120237",
        "start_month": "2025-02",
        "start_id": 3235,
        "end_month": "2025-02",
        "end_id": 8234,
    },
    "TAN": {
        "name": "Nguyễn Minh Tân",
        "member_id": "24120438",
        "start_month": "2024-07",
        "start_id": 12839,
        "end_month": "2024-07",
        "end_id": 17838,
    },
    "TRI": {
        "name": "Lương Trọng Trí",
        "member_id": "24120470",
        "start_month": "2024-06",
        "start_id": 17938,
        "end_month": "2024-07",
        "end_id": 2838,
    },
    "TRIET": {
        "name": "Nguyễn Minh Triết",
        "member_id": "24120474",
        "start_month": "2024-06",
        "start_id": 12938,
        "end_month": "2024-06",
        "end_id": 17937,
    },
}

if CURRENT_MEMBER not in MEMBERS_CONFIG:
    raise ValueError(f"CURRENT_MEMBER '{CURRENT_MEMBER}' không hợp lệ! Hãy chọn 1 trong: {list(MEMBERS_CONFIG.keys())}")

cfg = MEMBERS_CONFIG[CURRENT_MEMBER]
MEMBER_NAME: str = cfg["name"]
MEMBER_ID: str = cfg["member_id"]
START_MONTH: str = cfg["start_month"]
START_ID: int = cfg["start_id"]
END_MONTH: str = cfg["end_month"]
END_ID: int = cfg["end_id"]

# Thư mục và tham số chạy.
HF_REPO_ID: str = "nquocdat06/arxiv-harvester-data"
GIT_REPO_URL: str = "https://github.com/NguyenQuocDat06-AI/arxiv-harvester.git"
GIT_BRANCH: str = "develop"

WORKSPACE_DIR: str = "/kaggle/working/arxiv-harvester"
OUTPUT_BASE_DIR: str = "/kaggle/working/data"
SHARD_OUTPUT_DIR: str = f"{OUTPUT_BASE_DIR}/shards/{MEMBER_ID}"
COURSE_EXPORT_DIR: str = f"{OUTPUT_BASE_DIR}/course_export"

# Giãn request arXiv tối thiểu 3 giây.
RATE_LIMIT_SEC: float = 3.0
BATCH_SIZE: int = 50
PROFILE_INTERVAL_SEC: float = 0.5

print(f"Thành viên: {MEMBER_NAME} (MSSV: {MEMBER_ID})")
print(f"Dải phân công: {START_MONTH}.{START_ID:05d} -> {END_MONTH}.{END_ID:05d} (5,000 bài)")
print(f"Tốc độ giãn cách: {RATE_LIMIT_SEC}s / request ")
print(f"Thư mục kết quả: {SHARD_OUTPUT_DIR}")
print(f"Bảng xuất chuẩn môn học: {COURSE_EXPORT_DIR}")
print(f"Hugging Face Dataset: https://huggingface.co/datasets/{HF_REPO_ID}")


## 2. Cài môi trường

Cài dependencies, lấy source từ nhánh `develop` và kiểm tra kết nối arXiv/Semantic Scholar.


In [ ]:
import sys
import subprocess
import shutil
import requests
import time
from pathlib import Path

# Cài dependencies.
print("Đang cài đặt thư viện phụ thuộc...")
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q", "--upgrade",
    "pip", "huggingface_hub", "pyarrow", "requests", "psutil", "pytest", "sickle"
], check=True)

# Cập nhật source.
if os.path.exists(WORKSPACE_DIR):
    print(f"Cập nhật repo tại {WORKSPACE_DIR}...")
    subprocess.run(["git", "-C", WORKSPACE_DIR, "pull", "origin", GIT_BRANCH], check=True)
else:
    print(f"Đang clone repo từ {GIT_REPO_URL} (branch: {GIT_BRANCH})...")
    subprocess.run([
        "git", "clone", "-b", GIT_BRANCH, GIT_REPO_URL, WORKSPACE_DIR
    ], check=True)

# Thêm đường dẫn vào sys.path
if WORKSPACE_DIR not in sys.path:
    sys.path.insert(0, WORKSPACE_DIR)

os.chdir(WORKSPACE_DIR)
print(f"Thư mục làm việc hiện tại: {os.getcwd()}")

# Kiểm tra kết nối.
print("Kiểm tra kết nối mạng trực tiếp từ Kaggle:")

# Kiểm tra IP Kaggle
try:
    r = requests.get("https://api.ipify.org?format=json", timeout=5)
    print(f"Kaggle Outbound IP: {r.json().get('ip', 'Unknown')}")
except Exception:
    print("Kaggle Outbound IP: Đã kết nối mạng Internet.")

# Kiểm tra kết nối arXiv Mirror
try:
    r_arxiv = requests.head("https://export.arxiv.org", timeout=6)
    print(f"Kết nối arXiv Export Mirror: HTTP {r_arxiv.status_code} OK")
except Exception as exc:
    print(f"Kết nối arXiv Export Mirror: {exc}")

# Kiểm tra kết nối Semantic Scholar API
s2_headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
if os.environ.get("S2_API_KEY"):
    s2_headers["x-api-key"] = os.environ["S2_API_KEY"]

try:
    r_s2 = requests.get(
        "https://api.semanticscholar.org/graph/v1/paper/arXiv:2412.18848?fields=title",
        headers=s2_headers,
        timeout=6,
    )
    if r_s2.status_code == 200:
        print(f"Kết nối Semantic Scholar: HTTP 200 OK — '{r_s2.json().get('title', '')[:40]}...'")
    elif r_s2.status_code == 429:
        print("Semantic Scholar trả HTTP 429 (Rate Limit).")
        print("    Crawler xử lý retry theo cấu hình.")
    else:
        print(f"Semantic Scholar phản hồi: HTTP {r_s2.status_code}")
except Exception as exc:
    print(f"Kết nối Semantic Scholar: {exc}")


## 3. Khôi phục checkpoint

Tải checkpoint của thành viên từ Hugging Face. Bật `--resume` khi checkpoint local tồn tại.


In [ ]:
from huggingface_hub import HfApi, snapshot_download

has_remote_checkpoint = False
api = HfApi()

if HF_KEY:
    try:
        print(f"Đang kiểm tra checkpoint từ Hugging Face repo: {HF_REPO_ID}...")
        repo_files = api.list_repo_files(repo_id=HF_REPO_ID, repo_type="dataset", token=HF_KEY)
        prefix = f"shards/{MEMBER_ID}/_checkpoint"
        matching_files = [f for f in repo_files if f.startswith(prefix)]
        
        if matching_files:
            print(f"Tìm thấy {len(matching_files)} tệp checkpoint trên remote. Đang tải về để tiếp tục...")
            snapshot_download(
                repo_id=HF_REPO_ID,
                repo_type="dataset",
                allow_patterns=[f"shards/{MEMBER_ID}/*"],
                local_dir=OUTPUT_BASE_DIR,
                token=HF_KEY
            )
            # Chuyển checkpoint tải về vào shard.
            nested_shard = Path(OUTPUT_BASE_DIR) / "shards" / MEMBER_ID
            if nested_shard.exists():
                has_remote_checkpoint = True
                print(f"Đã khôi phục checkpoint: {MEMBER_NAME} ({MEMBER_ID}).")
        else:
            print("Chưa có checkpoint remote; chạy mới.")
    except Exception as exc:
        print(f"Không thể kiểm tra checkpoint từ xa ({exc}). Sẽ cào mới cục bộ.")
else:
    print("Chưa cấu hình HF_KEY, bỏ qua tải checkpoint từ remote.")


## 4. Chạy crawler

Truyền dải ID của thành viên, `--rate-limit-sec` và `--export-course`. `--resume` được bật khi có checkpoint. Log được in ra console và lưu vào `crawler.log`; shard được đồng bộ theo lô hoặc sau 10 phút.


In [ ]:
import os
import sys
import time
import subprocess
from pathlib import Path

# Resume khi có checkpoint.
checkpoint_state_file = Path(SHARD_OUTPUT_DIR) / "_checkpoint" / "state.json"
can_resume = has_remote_checkpoint or checkpoint_state_file.exists()

os.makedirs(SHARD_OUTPUT_DIR, exist_ok=True)
os.makedirs(COURSE_EXPORT_DIR, exist_ok=True)

cmd = [
    sys.executable, "-m", "src.crawler.main",
    "--member-id", str(MEMBER_ID),
    "--start-month", str(START_MONTH),
    "--start-id", str(START_ID),
    "--end-month", str(END_MONTH),
    "--end-id", str(END_ID),
    "--output-dir", str(SHARD_OUTPUT_DIR),
    "--batch-size", str(BATCH_SIZE),
    "--rate-limit-sec", str(RATE_LIMIT_SEC),
    "--profile-interval-sec", str(PROFILE_INTERVAL_SEC),
    "--export-course",
    "--course-export-dir", str(COURSE_EXPORT_DIR),
]

if can_resume:
    cmd.append("--resume")
    print("Chế độ thực thi: --resume (Tiếp tục từ checkpoint đã lưu)")
else:
    print("Chế độ thực thi: Cào mới từ đầu")

print("Lệnh thực thi:", " ".join(cmd))

# Chạy crawler và đọc log.
process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    cwd=WORKSPACE_DIR,
    env=os.environ.copy(),
)

# Mốc đồng bộ gần nhất.
last_sync_time = time.time()
sync_interval_sec = 600  # Tự động đẩy lên Hugging Face mỗi 10 phút hoặc sau mỗi batch 50 bài
log_file_path = Path(SHARD_OUTPUT_DIR) / "crawler.log"

try:
    with open(log_file_path, "a", encoding="utf-8") as log_file:
        for line in iter(process.stdout.readline, ''):
            print(line, end='', flush=True)
            log_file.write(line)
            log_file.flush()

            # Đồng bộ sau mỗi lô hoặc 10 phút.
            if ("Bài" in line and "xong" in line) or (time.time() - last_sync_time > sync_interval_sec):
                if HF_KEY and os.path.exists(SHARD_OUTPUT_DIR):
                    try:
                        api.upload_folder(
                            folder_path=SHARD_OUTPUT_DIR,
                            path_in_repo=f"shards/{MEMBER_ID}",
                            repo_id=HF_REPO_ID,
                            repo_type="dataset",
                            token=HF_KEY,
                            commit_message=f"Auto-sync progress shard {MEMBER_ID}",
                        )
                        last_sync_time = time.time()
                        print(f"\nĐã đồng bộ checkpoint lên Hugging Face.\n")
                    except Exception:
                        pass
    process.stdout.close()
    return_code = process.wait()
    if return_code != 0:
        print(f"Tiến trình kết thúc với mã lỗi: {return_code}")
    else:
        print("\nCrawler kết thúc.")
except KeyboardInterrupt:
    print("\nTiến trình bị dừng bởi người dùng.")


## 5. Đồng bộ Hugging Face

Upload shard cá nhân và hai bảng course export vào thư mục theo MSSV.


In [ ]:
import time

if HF_KEY:
    timestamp_str = time.strftime("%Y-%m-%d %H:%M:%S UTC", time.gmtime())
    
    # Upload shard.
    if os.path.exists(SHARD_OUTPUT_DIR):
        print(f"Đang đồng bộ thư mục phân vùng {SHARD_OUTPUT_DIR} lên Hugging Face...")
        try:
            api.upload_folder(
                folder_path=SHARD_OUTPUT_DIR,
                path_in_repo=f"shards/{MEMBER_ID}",
                repo_id=HF_REPO_ID,
                repo_type="dataset",
                token=HF_KEY,
                commit_message=f"Sync shard {MEMBER_ID} ({MEMBER_NAME}) at {timestamp_str}",
            )
            print(f"Đã đồng bộ shard: https://huggingface.co/datasets/{HF_REPO_ID}/tree/main/shards/{MEMBER_ID}")
        except Exception as exc:
            print(f"Lỗi khi tải shard lên Hugging Face: {exc}")

    # Upload course export.
    if os.path.exists(COURSE_EXPORT_DIR) and any(Path(COURSE_EXPORT_DIR).glob("*.parquet")):
        print(f"Đang đồng bộ bảng chuẩn môn học {COURSE_EXPORT_DIR} lên Hugging Face...")
        try:
            api.upload_folder(
                folder_path=COURSE_EXPORT_DIR,
                path_in_repo=f"course_export/{MEMBER_ID}",
                repo_id=HF_REPO_ID,
                repo_type="dataset",
                token=HF_KEY,
                commit_message=f"Sync course_export {MEMBER_ID} at {timestamp_str}",
            )
            print(f"Đã đồng bộ course export: https://huggingface.co/datasets/{HF_REPO_ID}/tree/main/course_export/{MEMBER_ID}")
        except Exception as exc:
            print(f"Lỗi khi tải course_export lên Hugging Face: {exc}")
else:
    print("Bỏ qua đồng bộ do chưa có HF_KEY.")


## 6. Kiểm tra kết quả

Đọc số dòng, dung lượng, thống kê và schema của các bảng đã xuất. Đối chiếu từng bảng với schema sáu cột.


In [ ]:
import pyarrow.parquet as pq
from pathlib import Path

print(f"Kết quả: {MEMBER_NAME} ({MEMBER_ID})")

p_shard = Path(SHARD_OUTPUT_DIR)
if p_shard.exists():
    print("\nTệp trong shard:")
    for f in sorted(p_shard.rglob("*")):
        if f.is_file():
            size_mb = f.stat().st_size / (1024 * 1024)
            print(f"  - {f.relative_to(p_shard)}: {size_mb:.2f} MB")

p_course = Path(COURSE_EXPORT_DIR)
if p_course.exists():
    print("\nCourse export:")
    papers_pq = p_course / "papers.parquet"
    refs_pq = p_course / "references.parquet"

    if papers_pq.exists():
        t_papers = pq.read_table(papers_pq)
        print(f"  * papers.parquet: {t_papers.num_rows:,} dòng, {t_papers.num_columns} cột")
        print(f"    Các cột: {t_papers.column_names}")
        expected_paper_cols = ['arxiv_id', 'title', 'authors', 'submission_date', 'revised_dates', 'venue']
        assert t_papers.column_names == expected_paper_cols, f"Sai schema papers! {t_papers.column_names} != {expected_paper_cols}"
        print("    papers.parquet: đủ sáu cột.")

    if refs_pq.exists():
        t_refs = pq.read_table(refs_pq)
        print(f"  * references.parquet: {t_refs.num_rows:,} dòng, {t_refs.num_columns} cột")
        print(f"    Các cột: {t_refs.column_names}")
        expected_ref_cols = ['arxiv_id', 'ref_arxiv_id', 'ref_title', 'ref_authors', 'ref_submission_date', 'ref_s2_id']
        assert t_refs.column_names == expected_ref_cols, f"Sai schema references! {t_refs.column_names} != {expected_ref_cols}"
        print("    references.parquet: đủ sáu cột.")
